# Paradigm 3 — LLM-Native Generative Recommenders

**Anchor paper**: GenRec (Ji et al., ECIR 2024) — Generative recommendation via instruction-tuned LLMs

**Architecture**: The user's interaction history is formatted as a natural-language prompt,
and the LLM directly generates the recommended item title. No separate retrieval or
collaborative filtering — the LLM *is* the recommender.

**Key insight from GenRec**: Fine-tuning a base LLM (LLaMA) with LoRA on sequential
recommendation data formatted as instruction-following examples enables the LLM to
learn item transition patterns and generate relevant next-item recommendations.

**Our implementation**:
- **Fine-tuned option**: QLoRA fine-tuning of LLaMA on Amazon Books (Colab Pro / GPU)
- **Zero-shot option**: Prompt-based recommendation using an API LLM (no fine-tuning)
- **Evaluation**: Generated titles are matched against the candidate pool via fuzzy matching

**Ranking metrics**: HR@K, NDCG@K (via beam search + title matching)  
**Explanation metrics**: N/A for pure generation; can be added by prompting for rationale

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import pickle
import numpy as np
import torch
from tutorial_utils import (
    evaluate_ranking, evaluate_explanations,
    save_results, save_predictions, LatencyTracker,
    DATA_DIR, RESULTS_DIR
)

with open(DATA_DIR / 'shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)

user_histories = shared['user_histories']
test_ground_truth = shared['test_ground_truth']
item_titles = shared['item_titles']
all_items = shared['all_items']
n_users = shared['n_users']
n_items = shared['n_items']

# Load GenRec-format test data
with open(DATA_DIR / 'genrec_test.json') as f:
    genrec_test = json.load(f)

print(f"Test examples: {len(genrec_test)}")
print(f"Sample: {json.dumps(genrec_test[0], indent=2)[:300]}")

## 1. Title matching utilities

The generative model outputs free-form text (a book title). We need to match
this against our item catalog to compute ranking metrics.

In [ ]:
from difflib import SequenceMatcher

# Build reverse index: title -> item_idx
title_to_idx = {}
for idx, title in item_titles.items():
    title_to_idx[title.lower().strip()] = idx


def match_title(generated_title: str, threshold: float = 0.8) -> int:
    """Match a generated title to our item catalog.
    Returns item_idx or -1 if no match found.
    """
    gen_lower = generated_title.lower().strip()

    # Exact match
    if gen_lower in title_to_idx:
        return title_to_idx[gen_lower]

    # Fuzzy match
    best_score = 0
    best_idx = -1
    for title, idx in title_to_idx.items():
        score = SequenceMatcher(None, gen_lower, title).ratio()
        if score > best_score:
            best_score = score
            best_idx = idx

    return best_idx if best_score >= threshold else -1


def match_titles_batch(generated_titles: list, threshold: float = 0.8) -> list:
    """Match a batch of generated titles. Returns list of item_idx (-1 for no match)."""
    return [match_title(t, threshold) for t in generated_titles]

## 2. Option A: QLoRA Fine-tuning (GPU required)

Fine-tune LLaMA-2-7B with QLoRA on the GenRec-format training data.
Requires Colab Pro (A100) or a local GPU with >= 16GB VRAM.

In [ ]:
# --- QLoRA fine-tuning ---
# Uncomment this section to fine-tune. Requires GPU.

# from transformers import (
#     AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
#     Trainer, BitsAndBytesConfig, DataCollatorForSeq2Seq,
# )
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from datasets import load_dataset
#
# MODEL_ID = "meta-llama/Llama-2-7b-hf"  # requires HF token
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16,
# )
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token
#
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, device_map="auto"
# )
# model = prepare_model_for_kbit_training(model)
#
# lora_config = LoraConfig(
#     r=8, lora_alpha=16, lora_dropout=0.05,
#     target_modules=["q_proj", "v_proj"],
#     task_type="CAUSAL_LM",
# )
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()
#
# # Format prompt (matches GenRec's training format)
# CUTOFF_LEN = 512
#
# def tokenize_example(example):
#     prompt = f"{example['instruction']}\n\n### input:\n{example['input']}\n\n### Response:\n"
#     full = prompt + example['output'] + tokenizer.eos_token
#     tokenized = tokenizer(full, truncation=True, max_length=CUTOFF_LEN, padding="max_length")
#     prompt_len = len(tokenizer(prompt, truncation=True, max_length=CUTOFF_LEN)["input_ids"])
#     tokenized["labels"] = [-100] * prompt_len + tokenized["input_ids"][prompt_len:]
#     return tokenized
#
# train_data = load_dataset("json", data_files=str(DATA_DIR / "genrec_train.json"), split="train")
# train_data = train_data.map(tokenize_example, remove_columns=train_data.column_names)
#
# training_args = TrainingArguments(
#     output_dir=str(RESULTS_DIR / "genrec_qlora"),
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=8,
#     learning_rate=3e-4,
#     fp16=True,
#     save_strategy="epoch",
#     logging_steps=50,
#     warmup_steps=100,
# )
#
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_data,
#     data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
# )
# trainer.train()
# model.save_pretrained(str(RESULTS_DIR / "genrec_qlora" / "final"))

## 3. Option B: Zero-shot API-based generation

In [ ]:
GENERATION_PROMPT = """{instruction}

### input:
{input}

### Response:"""


def format_generation_prompt(example):
    return GENERATION_PROMPT.format(
        instruction=example['instruction'],
        input=example['input'],
    )

In [ ]:
# --- Option B: API-based zero-shot ---
# import openai
# client = openai.OpenAI(api_key="YOUR_KEY")
#
# def generate_recommendation_api(example, n_candidates=10):
#     prompt = format_generation_prompt(example)
#     prompt += "\nProvide your top 10 recommendations as a numbered list, one per line."
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=500,
#         temperature=0.3,
#     )
#     text = response.choices[0].message.content.strip()
#     # Parse numbered list
#     import re
#     titles = re.findall(r'\d+\.\s*(.+)', text)
#     return [t.strip().strip('"') for t in titles[:n_candidates]]

In [ ]:
# --- Placeholder generator (for testing the pipeline without LLM) ---

def generate_recommendation_placeholder(example, n_candidates=10):
    """Placeholder: returns random items from the catalog.
    Replace with generate_recommendation_api or fine-tuned model inference.
    """
    rng = np.random.RandomState(hash(example['input']) % (2**31))
    sampled = rng.choice(list(item_titles.values()), size=min(n_candidates, len(item_titles)), replace=False)
    return list(sampled)

## 4. Run generation + evaluation

In [ ]:
# Build mapping from test examples to user indices
# The genrec_test examples align with test_ground_truth by construction
test_users = list(test_ground_truth.keys())

tracker = LatencyTracker()
predictions = {}
explanations = {}

# Choose generator: generate_recommendation_api or generate_recommendation_placeholder
generate_fn = generate_recommendation_placeholder

for i, (example, user_idx) in enumerate(zip(genrec_test, test_users)):
    if i >= len(test_users):
        break

    tracker.start()
    generated_titles = generate_fn(example, n_candidates=10)
    matched_items = match_titles_batch(generated_titles, threshold=0.7)
    # Filter out unmatched (-1) items
    valid_items = [it for it in matched_items if it >= 0]
    predictions[user_idx] = valid_items
    tracker.stop()

    # Use the generated text as explanation
    if valid_items:
        explanations[user_idx] = {
            "item_idx": valid_items[0],
            "text": f"Based on your reading of {example['input'][:100]}..., "
                    f"I recommend '{generated_titles[0]}' as your next read."
        }

    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1}/{min(len(genrec_test), len(test_users))}")

# Match rate
n_with_predictions = sum(1 for v in predictions.values() if v)
print(f"\nUsers with matched predictions: {n_with_predictions}/{len(predictions)}")
print(f"Match rate: {n_with_predictions/len(predictions):.2%}")

In [ ]:
ranking_results = evaluate_ranking(predictions, test_ground_truth, k_values=[5, 10, 20])
latency = tracker.summary()

print("Ranking results:")
for metric, val in ranking_results.items():
    print(f"  {metric}: {val:.4f}")
print(f"\nLatency: {latency['mean_latency_ms']:.2f} ms/recommendation")

# Explanation evaluation (on subset with explanations)
explanation_results = {}
if explanations:
    explanation_results = evaluate_explanations(
        {u: explanations[u] for u in list(explanations.keys())[:100]},
        user_histories, item_titles, set(item_titles.values())
    )
    print("\nExplanation quality:")
    for metric, val in explanation_results.items():
        print(f"  {metric}: {val:.4f}")

## 5. Save results

In [ ]:
results = {
    "paradigm": "generative",
    "model": "LLM generative (GenRec-style)",
    "anchor_paper": "GenRec (Ji et al., ECIR 2024)",
    "ranking": ranking_results,
    "explanation": explanation_results,
    "system": {"latency": latency},
}

save_results("generative", results)
save_predictions("generative", predictions)
print("Done.")